# 16 · Try-it answers for the dashboard (D-068)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**). About 15 minutes.

The dashboard's *Try it* section shows five openly licensed photos (in
`dashboard/public/samples/`, never VizWiz images) with the fine-tuned model's answers.
This notebook computes those answers once, with the served setup: vLLM + the E5
adapter + the gateway (threshold 0.10, PNG photos). The dashboard then shows them with
no GPU running.

1. Settings, Drive, code (~1 min)
2. Model-server environment (~5 min)
3. Start vLLM with E5 and the gateway (~4 min)
4. Ask about the five photos (~10 s) and save `answers.json`

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code

In [ ]:
import json
import os
import subprocess
import time
import urllib.request

from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
ADAPTER = f"{DRIVE_ROOT}/checkpoints/e5-multi-lora/adapter"
OUT = f"{DRIVE_ROOT}/runs/demo"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
MODEL = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_URL, GATEWAY_URL = "http://127.0.0.1:8000", "http://127.0.0.1:8090"  # Colab itself uses 8080


def sh(cmd, check=True, **env):
    """Run a shell command, stream its output, and raise if it fails (unless check=False)."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed (exit {code}). Stop here and paste the output above.")
    return code


def wait_for(url, seconds, proc=None, log=None):
    t0 = time.time()
    while time.time() - t0 < seconds:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                return json.loads(r.read()), time.time() - t0
        except Exception:
            if proc is not None and proc.poll() is not None:
                break
            time.sleep(1)
    if log:
        sh(f"tail -n 40 {log}", check=False)
    raise RuntimeError(f"{url} did not come up. Paste the log lines above.")


os.environ["HF_HOME"] = "/content/hf_cache"
# A folder created under /content/drive before mounting blocks the mount: remove empty ones only.
if not os.path.ismount("/content/drive"):
    subprocess.run("find /content/drive -depth -type d -empty -delete", shell=True, check=False)
drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)  # only after Drive is mounted
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "E5 adapter not found on Drive"
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
sh('pkill -f "[v]llm serve"; pkill -f "[v]izwiz.serve.gateway"; sleep 2', check=False)

**Paste back:** the commit line and the GPU line (must say NVIDIA L4).

## 2 · Model-server environment (~5 min)
The same pinned environment as notebooks 13–15.

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e '{REPO_DIR}[serve]' -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")

## 3 · Start vLLM with E5, then the gateway (~4 min)
The gateway uses the committed `configs/serve.yaml`: E5 for all three stages, threshold 0.10, PNG photos.

In [ ]:
%%time
env = {
    **os.environ,
    "PATH": f"{VENV}/bin:" + os.environ["PATH"],
    "VLLM_USE_FLASHINFER_SAMPLER": "0",  # R-002, D-020
}
vllm_cmd = (
    f"{VENV}/bin/vllm serve {MODEL} --dtype bfloat16 --max-model-len 4096 --gpu-memory-utilization 0.85 "
    f"--enable-lora --lora-modules e5-multi-lora={ADAPTER} --max-lora-rank 16 --max-loras 4 "
    f"--limit-mm-per-prompt '{{\"image\": 1}}' --generation-config vllm --seed 0 --port 8000"
)
vllm_proc = subprocess.Popen(
    vllm_cmd, shell=True, env=env, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT
)
_, secs = wait_for(f"{MODEL_URL}/v1/models", 900, vllm_proc, "/content/vllm.log")
print(f"model server up after {secs:.0f} s")
gw_proc = subprocess.Popen(
    f"{VENV}/bin/uvicorn --factory vizwiz.serve.gateway:create_app --host 127.0.0.1 --port 8090",
    shell=True,
    env={**env, "VIZWIZ_MODEL_URL": MODEL_URL},
    cwd=REPO_DIR,
    stdout=open("/content/gateway.log", "w"),
    stderr=subprocess.STDOUT,
)
health, _ = wait_for(f"{GATEWAY_URL}/health", 300, gw_proc, "/content/gateway.log")
print("GATEWAY:", health)

**Paste back:** the `model server up` line and the `GATEWAY` line (status must be `ok`).

## 4 · Ask about the five photos and save
One request per photo, one at a time. Saved to `MyDrive/vizwiz/runs/demo/answers.json`.

In [ ]:
sh(
    f"cd {REPO_DIR} && {PY} -m vizwiz.serve.demo --gateway {GATEWAY_URL} "
    f"--samples dashboard/public/samples/samples.json --out {OUT}/answers.json"
)
sh('pkill -f "[v]llm serve"; pkill -f "[v]izwiz.serve.gateway"', check=False)
drive.flush_and_unmount()
print("Saved, servers stopped, Drive flushed. Safe to disconnect.")

**Paste back:** all `DEMO` lines.

Then download `MyDrive/vizwiz/runs/demo/answers.json` and put it in the repo at
`results/demo/answers.json` (or paste its contents back), and Runtime →
**Disconnect and delete runtime**.